<a href="https://colab.research.google.com/github/sharath7-dev/robust-multilingual-intent-classification/blob/main/notebooks/phase1_dataset_verification_tfidf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Robust and Calibrated Multilingual Intent Classification

## Phase 1 — Dataset Verification and TF-IDF Baseline

### Goal
Verify the multilingual dataset, construct leakage-safe known and unseen intent groups, and establish a reproducible classical baseline without evaluating the test set.

### Configuration
- Dataset: MASSIVE 1.1
- Languages: English (`en-US`), Hindi (`hi-IN`), Kannada (`kn-IN`)
- Released examples: 49,563
- Original intents: 60
- Known intents used for training: 50
- Validation-OOS intents: 5
- Test-OOS intents: 5
- Random seed: 42

### Verified data integrity
- Official train, development, and test partitions preserved
- No missing utterances or intent labels
- No duplicate IDs within a locale
- Parallel examples remain in the same partition and intent across languages
- No held-out OOS intent appears in training

### TF-IDF validation baseline
The baseline combines word and character TF-IDF features with logistic regression.

| Evaluation group | Accuracy | Macro-F1 |
|---|---:|---:|
| Overall validation | 0.8369 | 0.8210 |
| English | 0.8594 | 0.8394 |
| Hindi | 0.8308 | 0.8140 |
| Kannada | 0.8205 | 0.8043 |

Training completed in 61.8 seconds. The solver converged after 115 iterations. The final test set has not been evaluated.

In [1]:
import sys
import platform

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())

try:
    import torch
    print("PyTorch:", torch.__version__)
    print("GPU available:", torch.cuda.is_available())
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
except Exception as error:
    print("PyTorch check failed:", error)

Python: 3.13.16
Platform: Linux-6.6.122+-x86_64-with-glibc2.39
PyTorch: 2.11.0+cu130
GPU available: True
GPU: Tesla T4


In [3]:
%cd /content
!git clone https://github.com/sharath7-dev/robust-multilingual-intent-classification.git
%cd /content/robust-multilingual-intent-classification
!git status

/content
Cloning into 'robust-multilingual-intent-classification'...
remote: Enumerating objects: 4, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 4 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (4/4), done.
/content/robust-multilingual-intent-classification
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [4]:
from pathlib import Path

repo = Path("/content/robust-multilingual-intent-classification")

folders = [
    "notebooks",
    "src",
    "configs",
    "results/metrics",
    "results/figures",
    "docs",
    "ui",
]

for folder in folders:
    path = repo / folder
    path.mkdir(parents=True, exist_ok=True)
    (path / ".gitkeep").touch()

extra_gitignore = """

# Project data and trained artifacts
data/
models/
checkpoints/
*.pt
*.pth
*.ckpt
*.safetensors
*.bin

# Secrets and experiment services
.env
.env.*
wandb/
mlruns/

# Generated temporary outputs
*.log
.ipynb_checkpoints/
"""

gitignore = repo / ".gitignore"
existing = gitignore.read_text(encoding="utf-8") if gitignore.exists() else ""

if "# Project data and trained artifacts" not in existing:
    gitignore.write_text(existing.rstrip() + extra_gitignore, encoding="utf-8")

print("Created project structure:")
for folder in folders:
    print(" -", folder)

print("\nGit status:")

Created project structure:
 - notebooks
 - src
 - configs
 - results/metrics
 - results/figures
 - docs
 - ui

Git status:


In [5]:
!git status --short

 M .gitignore
?? configs/
?? docs/
?? notebooks/
?? results/
?? src/
?? ui/


In [6]:
from google.colab import userdata
from pathlib import Path
import os
import subprocess
import tempfile

repo = Path("/content/robust-multilingual-intent-classification")
token = userdata.get("GITHUB_TOKEN")

if not token:
    raise RuntimeError("GITHUB_TOKEN is unavailable. Check Notebook access in Colab Secrets.")

def run_git(*args, env=None):
    return subprocess.run(
        ["git", *args],
        cwd=repo,
        env=env,
        text=True,
        check=True,
    )

run_git("config", "user.name", "sharath7-dev")
run_git("config", "user.email", "sharath7-dev@users.noreply.github.com")

run_git("add", ".gitignore", "configs", "docs", "notebooks", "results", "src", "ui")
run_git("commit", "-m", "chore: initialize project structure and gitignore")

askpass_path = None

try:
    with tempfile.NamedTemporaryFile(
        mode="w", suffix=".sh", delete=False, encoding="utf-8"
    ) as askpass:
        askpass.write(
            '#!/bin/sh\n'
            'case "$1" in\n'
            '  *Username*) echo "x-access-token" ;;\n'
            '  *) echo "$GITHUB_TOKEN" ;;\n'
            'esac\n'
        )
        askpass_path = askpass.name

    os.chmod(askpass_path, 0o700)

    push_env = os.environ.copy()
    push_env["GITHUB_TOKEN"] = token
    push_env["GIT_ASKPASS"] = askpass_path
    push_env["GIT_TERMINAL_PROMPT"] = "0"

    run_git("push", "origin", "main", env=push_env)

finally:
    if askpass_path and os.path.exists(askpass_path):
        os.remove(askpass_path)

print("\nLatest commit:")
run_git("log", "-1", "--oneline")

print("\nWorking tree:")
run_git("status", "--short")


Latest commit:

Working tree:


CompletedProcess(args=['git', 'status', '--short'], returncode=0)

In [7]:
import subprocess
from pathlib import Path

repo = Path("/content/robust-multilingual-intent-classification")

def show_git(*args):
    result = subprocess.run(
        ["git", *args],
        cwd=repo,
        text=True,
        capture_output=True,
        check=True,
    )
    print(result.stdout.strip() or "(no output)")

print("Latest local commit:")
show_git("log", "-1", "--oneline")

print("\nRemote tracking status:")
show_git("status", "--short", "--branch")

Latest local commit:
087b5c7 chore: initialize project structure and gitignore

Remote tracking status:
## main...origin/main


In [8]:
from pathlib import Path
from urllib.request import urlretrieve

repo = Path("/content/robust-multilingual-intent-classification")
download_dir = repo / "data"
download_dir.mkdir(exist_ok=True)

url = "https://amazon-massive-nlu-dataset.s3.amazonaws.com/amazon-massive-dataset-1.1.tar.gz"
archive = download_dir / "amazon-massive-dataset-1.1.tar.gz"

if not archive.exists():
    print("Downloading MASSIVE 1.1...")
    urlretrieve(url, archive)
else:
    print("Archive already exists.")

size_mb = archive.stat().st_size / (1024 * 1024)

print("File:", archive.name)
print(f"Size: {size_mb:.2f} MB")
print("Download verified:", archive.exists() and archive.stat().st_size > 0)

File: amazon-massive-dataset-1.1.tar.gz
Size: 38.39 MB
Download verified: True


In [9]:
import tarfile
from pathlib import Path

repo = Path("/content/robust-multilingual-intent-classification")
data_dir = repo / "data"
archive = data_dir / "amazon-massive-dataset-1.1.tar.gz"

with tarfile.open(archive, "r:gz") as tar:
    tar.extractall(data_dir, filter="data")

required_files = {}

for locale in ["en-US", "hi-IN", "kn-IN"]:
    matches = list(data_dir.rglob(f"{locale}.jsonl"))
    required_files[locale] = matches

print("Required language files:")

for locale, matches in required_files.items():
    if len(matches) == 1:
        print(f"{locale}: FOUND — {matches[0]}")
    else:
        print(f"{locale}: ERROR — found {len(matches)} matching files")

Required language files:
en-US: FOUND — /content/robust-multilingual-intent-classification/data/1.1/data/en-US.jsonl
hi-IN: FOUND — /content/robust-multilingual-intent-classification/data/1.1/data/hi-IN.jsonl
kn-IN: FOUND — /content/robust-multilingual-intent-classification/data/1.1/data/kn-IN.jsonl


In [10]:
import json
import pandas as pd
from pathlib import Path

data_root = Path(
    "/content/robust-multilingual-intent-classification/data/1.1/data"
)

locales = ["en-US", "hi-IN", "kn-IN"]
frames = []

for locale in locales:
    file_path = data_root / f"{locale}.jsonl"

    with file_path.open("r", encoding="utf-8") as file:
        records = [json.loads(line) for line in file]

    df_locale = pd.DataFrame(records)
    frames.append(df_locale)

    print(f"\n--- {locale} ---")
    print("Rows:", len(df_locale))
    print("Partitions:", df_locale["partition"].value_counts().to_dict())
    print("Unique intents:", df_locale["intent"].nunique())
    print("Missing utterances:", df_locale["utt"].isna().sum())
    print("Missing intents:", df_locale["intent"].isna().sum())
    print("Duplicate IDs:", df_locale["id"].duplicated().sum())
    print("Sample:", df_locale["utt"].iloc[0])

df = pd.concat(frames, ignore_index=True)

print("\nCombined rows:", len(df))
print("Columns:", df.columns.tolist())
print("Locales:", sorted(df["locale"].unique().tolist()))


--- en-US ---
Rows: 16521
Partitions: {'train': 11514, 'test': 2974, 'dev': 2033}
Unique intents: 60
Missing utterances: 0
Missing intents: 0
Duplicate IDs: 0
Sample: wake me up at five am this week

--- hi-IN ---
Rows: 16521
Partitions: {'train': 11514, 'test': 2974, 'dev': 2033}
Unique intents: 60
Missing utterances: 0
Missing intents: 0
Duplicate IDs: 0
Sample: मुझे इस सप्ताह सुबह पांच बजे जगा दो

--- kn-IN ---
Rows: 16521
Partitions: {'train': 11514, 'test': 2974, 'dev': 2033}
Unique intents: 60
Missing utterances: 0
Missing intents: 0
Duplicate IDs: 0
Sample: ಈ ವಾರ ಬೆಳಗ್ಗೆ ಐದು ಗಂಟೆಗೆ ನನ್ನನ್ನು ಎದ್ದೇಳಿ

Combined rows: 49563
Columns: ['id', 'locale', 'partition', 'scenario', 'intent', 'utt', 'annot_utt', 'worker_id', 'slot_method', 'judgments']
Locales: ['en-US', 'hi-IN', 'kn-IN']


In [11]:
# Each shared utterance ID should remain in the same partition
# and have the same intent across all three languages.

id_checks = df.groupby("id").agg(
    locale_count=("locale", "nunique"),
    partition_count=("partition", "nunique"),
    intent_count=("intent", "nunique"),
)

incomplete_ids = (id_checks["locale_count"] != 3).sum()
partition_conflicts = (id_checks["partition_count"] != 1).sum()
intent_conflicts = (id_checks["intent_count"] != 1).sum()

print("Unique shared IDs:", len(id_checks))
print("IDs missing one or more selected languages:", incomplete_ids)
print("IDs assigned to multiple partitions:", partition_conflicts)
print("IDs assigned to multiple intents:", intent_conflicts)

passed = (
    incomplete_ids == 0
    and partition_conflicts == 0
    and intent_conflicts == 0
)

print("Parallel split verification passed:", passed)

Unique shared IDs: 16521
IDs missing one or more selected languages: 0
IDs assigned to multiple partitions: 0
IDs assigned to multiple intents: 0
Parallel split verification passed: True


In [12]:
import json
import random
from pathlib import Path

repo = Path("/content/robust-multilingual-intent-classification")

all_intents = sorted(df["intent"].unique().tolist())

rng = random.Random(42)
held_out_intents = sorted(rng.sample(all_intents, 10))

validation_oos_intents = held_out_intents[:5]
test_oos_intents = held_out_intents[5:]
known_intents = sorted(set(all_intents) - set(held_out_intents))

intent_split = {
    "seed": 42,
    "known_intents": known_intents,
    "validation_oos_intents": validation_oos_intents,
    "test_oos_intents": test_oos_intents,
}

config_path = repo / "configs" / "intent_split.json"
config_path.write_text(
    json.dumps(intent_split, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Known intents:", len(known_intents))
print("Validation OOS intents:", validation_oos_intents)
print("Test OOS intents:", test_oos_intents)
print("Overlap between OOS groups:",
      set(validation_oos_intents) & set(test_oos_intents))
print("Saved:", config_path)

Known intents: 50
Validation OOS intents: ['alarm_remove', 'audio_volume_up', 'calendar_query', 'calendar_remove', 'email_addcontact']
Test OOS intents: ['email_query', 'email_sendemail', 'play_music', 'qa_stock', 'transport_taxi']
Overlap between OOS groups: set()
Saved: /content/robust-multilingual-intent-classification/configs/intent_split.json


In [13]:
known_set = set(known_intents)
validation_oos_set = set(validation_oos_intents)
test_oos_set = set(test_oos_intents)

train_known = df[
    (df["partition"] == "train") &
    (df["intent"].isin(known_set))
].copy()

validation_known = df[
    (df["partition"] == "dev") &
    (df["intent"].isin(known_set))
].copy()

validation_oos = df[
    (df["partition"] == "dev") &
    (df["intent"].isin(validation_oos_set))
].copy()

test_known = df[
    (df["partition"] == "test") &
    (df["intent"].isin(known_set))
].copy()

test_oos = df[
    (df["partition"] == "test") &
    (df["intent"].isin(test_oos_set))
].copy()

splits = {
    "train_known": train_known,
    "validation_known": validation_known,
    "validation_oos": validation_oos,
    "test_known": test_known,
    "test_oos": test_oos,
}

print("Split sizes:")
for name, split_df in splits.items():
    print(f"\n{name}: {len(split_df)}")
    print(split_df["locale"].value_counts().sort_index().to_dict())

training_leakage = set(train_known["intent"]) & (
    validation_oos_set | test_oos_set
)

print("\nHeld-out intents found in training:", training_leakage)
print("Training has exactly 50 intents:", train_known["intent"].nunique() == 50)
print("Verification passed:",
      not training_leakage and train_known["intent"].nunique() == 50)

Split sizes:

train_known: 26193
{'en-US': 8731, 'hi-IN': 8731, 'kn-IN': 8731}

validation_known: 4629
{'en-US': 1543, 'hi-IN': 1543, 'kn-IN': 1543}

validation_oos: 540
{'en-US': 180, 'hi-IN': 180, 'kn-IN': 180}

test_known: 6831
{'en-US': 2277, 'hi-IN': 2277, 'kn-IN': 2277}

test_oos: 1374
{'en-US': 458, 'hi-IN': 458, 'kn-IN': 458}

Held-out intents found in training: set()
Training has exactly 50 intents: True
Verification passed: True


In [14]:
import json
from pathlib import Path

repo = Path("/content/robust-multilingual-intent-classification")

verification_report = {
    "dataset": "MASSIVE 1.1",
    "source_archive": "amazon-massive-dataset-1.1.tar.gz",
    "archive_size_mb": 38.39,
    "locales": ["en-US", "hi-IN", "kn-IN"],
    "rows_per_locale": 16521,
    "combined_rows": 49563,
    "official_split_counts_per_locale": {
        "train": 11514,
        "dev": 2033,
        "test": 2974,
    },
    "unique_intents": 60,
    "missing_utterances": 0,
    "missing_intents": 0,
    "duplicate_ids_per_locale": 0,
    "parallel_split_conflicts": 0,
    "parallel_intent_conflicts": 0,
    "known_intents": 50,
    "validation_oos_intents": validation_oos_intents,
    "test_oos_intents": test_oos_intents,
    "experimental_split_sizes": {
        name: {
            "total": len(split_df),
            "by_locale": split_df["locale"].value_counts().sort_index().to_dict(),
        }
        for name, split_df in splits.items()
    },
    "training_oos_leakage": False,
    "verification_passed": True,
}

report_path = repo / "results" / "metrics" / "dataset_verification.json"
report_path.write_text(
    json.dumps(verification_report, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print(report_path.read_text(encoding="utf-8"))
print("\nGit status:")

{
  "dataset": "MASSIVE 1.1",
  "source_archive": "amazon-massive-dataset-1.1.tar.gz",
  "archive_size_mb": 38.39,
  "locales": [
    "en-US",
    "hi-IN",
    "kn-IN"
  ],
  "rows_per_locale": 16521,
  "combined_rows": 49563,
  "official_split_counts_per_locale": {
    "train": 11514,
    "dev": 2033,
    "test": 2974
  },
  "unique_intents": 60,
  "missing_utterances": 0,
  "missing_intents": 0,
  "duplicate_ids_per_locale": 0,
  "parallel_split_conflicts": 0,
  "parallel_intent_conflicts": 0,
  "known_intents": 50,
  "validation_oos_intents": [
    "alarm_remove",
    "audio_volume_up",
    "calendar_query",
    "calendar_remove",
    "email_addcontact"
  ],
  "test_oos_intents": [
    "email_query",
    "email_sendemail",
    "play_music",
    "qa_stock",
    "transport_taxi"
  ],
  "experimental_split_sizes": {
    "train_known": {
      "total": 26193,
      "by_locale": {
        "en-US": 8731,
        "hi-IN": 8731,
        "kn-IN": 8731
      }
    },
    "validation_known": {

In [15]:
!git status --short

?? configs/intent_split.json
?? results/metrics/dataset_verification.json


In [16]:
from google.colab import userdata
from pathlib import Path
import os
import subprocess
import tempfile

repo = Path("/content/robust-multilingual-intent-classification")
token = userdata.get("GITHUB_TOKEN")

subprocess.run(
    ["git", "add",
     "configs/intent_split.json",
     "results/metrics/dataset_verification.json"],
    cwd=repo,
    check=True,
)

subprocess.run(
    ["git", "commit", "-m",
     "data: verify MASSIVE multilingual splits and freeze OOS intents"],
    cwd=repo,
    check=True,
)

askpass_path = None

try:
    with tempfile.NamedTemporaryFile(
        mode="w", suffix=".sh", delete=False, encoding="utf-8"
    ) as askpass:
        askpass.write(
            '#!/bin/sh\n'
            'case "$1" in\n'
            '  *Username*) echo "x-access-token" ;;\n'
            '  *) echo "$GITHUB_TOKEN" ;;\n'
            'esac\n'
        )
        askpass_path = askpass.name

    os.chmod(askpass_path, 0o700)

    env = os.environ.copy()
    env["GITHUB_TOKEN"] = token
    env["GIT_ASKPASS"] = askpass_path
    env["GIT_TERMINAL_PROMPT"] = "0"

    subprocess.run(
        ["git", "push", "origin", "main"],
        cwd=repo,
        env=env,
        check=True,
    )

finally:
    if askpass_path and os.path.exists(askpass_path):
        os.remove(askpass_path)

latest = subprocess.run(
    ["git", "log", "-1", "--oneline"],
    cwd=repo,
    text=True,
    capture_output=True,
    check=True,
)

status = subprocess.run(
    ["git", "status", "--short", "--branch"],
    cwd=repo,
    text=True,
    capture_output=True,
    check=True,
)

print("Latest commit:", latest.stdout.strip())
print("Status:", status.stdout.strip())

Latest commit: a7accaa data: verify MASSIVE multilingual splits and freeze OOS intents
Status: ## main...origin/main


In [17]:
import subprocess
from pathlib import Path

repo = Path("/content/robust-multilingual-intent-classification")

pull = subprocess.run(
    ["git", "pull", "--ff-only", "origin", "main"],
    cwd=repo,
    text=True,
    capture_output=True,
    check=True,
)

status = subprocess.run(
    ["git", "status", "--short", "--branch"],
    cwd=repo,
    text=True,
    capture_output=True,
    check=True,
)

print(pull.stdout.strip())
print(status.stdout.strip())

Updating a7accaa..368eab3
Fast-forward
 notebooks/phase1_dataset_verification_tfidf.ipynb | 1025 +++++++++++++++++++++
 1 file changed, 1025 insertions(+)
 create mode 100644 notebooks/phase1_dataset_verification_tfidf.ipynb
## main...origin/main


In [18]:
import time
import numpy as np
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

tfidf_features = FeatureUnion([
    (
        "word",
        TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2),
            min_df=2,
            max_features=30_000,
            sublinear_tf=True,
        ),
    ),
    (
        "character",
        TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            min_df=2,
            max_features=50_000,
            sublinear_tf=True,
        ),
    ),
])

tfidf_model = Pipeline([
    ("features", tfidf_features),
    (
        "classifier",
        LogisticRegression(
            solver="saga",
            max_iter=300,
            C=4.0,
            random_state=42,
            n_jobs=-1,
        ),
    ),
])

start_time = time.time()

tfidf_model.fit(
    train_known["utt"],
    train_known["intent"],
)

training_seconds = time.time() - start_time

validation_predictions = tfidf_model.predict(validation_known["utt"])

validation_accuracy = accuracy_score(
    validation_known["intent"],
    validation_predictions,
)

validation_macro_f1 = f1_score(
    validation_known["intent"],
    validation_predictions,
    average="macro",
)

print(f"Training time: {training_seconds:.1f} seconds")
print(f"Validation accuracy: {validation_accuracy:.4f}")
print(f"Validation macro-F1: {validation_macro_f1:.4f}")

print("\nPer-language validation results:")

for locale in ["en-US", "hi-IN", "kn-IN"]:
    mask = validation_known["locale"] == locale

    locale_accuracy = accuracy_score(
        validation_known.loc[mask, "intent"],
        validation_predictions[mask],
    )

    locale_macro_f1 = f1_score(
        validation_known.loc[mask, "intent"],
        validation_predictions[mask],
        average="macro",
    )

    print(
        f"{locale}: accuracy={locale_accuracy:.4f}, "
        f"macro-F1={locale_macro_f1:.4f}"
    )

Training time: 61.8 seconds
Validation accuracy: 0.8369
Validation macro-F1: 0.8210

Per-language validation results:
en-US: accuracy=0.8594, macro-F1=0.8394
hi-IN: accuracy=0.8308, macro-F1=0.8140
kn-IN: accuracy=0.8205, macro-F1=0.8043


In [19]:
import json
import joblib
import sklearn
from pathlib import Path

repo = Path("/content/robust-multilingual-intent-classification")

per_language_results = {}

for locale in ["en-US", "hi-IN", "kn-IN"]:
    mask = validation_known["locale"] == locale

    per_language_results[locale] = {
        "accuracy": float(
            accuracy_score(
                validation_known.loc[mask, "intent"],
                validation_predictions[mask],
            )
        ),
        "macro_f1": float(
            f1_score(
                validation_known.loc[mask, "intent"],
                validation_predictions[mask],
                average="macro",
            )
        ),
    }

baseline_metrics = {
    "model": "word-and-character TF-IDF with logistic regression",
    "random_seed": 42,
    "scikit_learn_version": sklearn.__version__,
    "training_examples": len(train_known),
    "validation_examples": len(validation_known),
    "training_seconds": training_seconds,
    "validation_accuracy": float(validation_accuracy),
    "validation_macro_f1": float(validation_macro_f1),
    "per_language": per_language_results,
    "test_set_evaluated": False,
}

metrics_path = repo / "results" / "metrics" / "tfidf_validation_metrics.json"
metrics_path.write_text(
    json.dumps(baseline_metrics, indent=2),
    encoding="utf-8",
)

model_dir = repo / "models"
model_dir.mkdir(exist_ok=True)
model_path = model_dir / "tfidf_baseline.joblib"
joblib.dump(tfidf_model, model_path)

max_iterations_used = int(
    tfidf_model.named_steps["classifier"].n_iter_.max()
)

print("Metrics saved:", metrics_path)
print("Model saved outside Git tracking:", model_path)
print(f"Model size: {model_path.stat().st_size / (1024**2):.2f} MB")
print("Maximum solver iterations used:", max_iterations_used)
print("Reached configured 300-iteration limit:", max_iterations_used >= 300)
print("\nGit status:")

Metrics saved: /content/robust-multilingual-intent-classification/results/metrics/tfidf_validation_metrics.json
Model saved outside Git tracking: /content/robust-multilingual-intent-classification/models/tfidf_baseline.joblib
Model size: 27.56 MB
Maximum solver iterations used: 115
Reached configured 300-iteration limit: False

Git status:


In [20]:
%cd /content/robust-multilingual-intent-classification
!git status --short --untracked-files=all
!git check-ignore -v results/metrics/tfidf_validation_metrics.json || echo "Metrics file is not ignored"

/content/robust-multilingual-intent-classification
?? results/metrics/tfidf_validation_metrics.json
Metrics file is not ignored


In [23]:
%%writefile /content/robust-multilingual-intent-classification/README.md
# Robust and Calibrated Multilingual Intent Classification

A reproducible NLP project studying multilingual intent classification under spelling noise, transliterated input, unseen intents, and uncertainty-aware rejection.

## Current scope

Languages:

- English (`en-US`)
- Hindi (`hi-IN`)
- Kannada (`kn-IN`)

## Dataset

The project uses the public [MASSIVE 1.1 dataset](https://github.com/alexa/massive), licensed under CC BY 4.0. Dataset files are downloaded when required and are not stored in this repository.

Verified statistics:

- 49,563 examples across the selected languages
- 60 original intents
- 50 known training intents
- 5 validation-OOS intents
- 5 separate test-OOS intents
- Official dataset partitions preserved
- No held-out intent leakage into training

The frozen intent groups are stored in [`configs/intent_split.json`](configs/intent_split.json).

## Phase 1 verified baseline

Model: word-and-character TF-IDF with logistic regression.

| Evaluation group | Accuracy | Macro-F1 |
|---|---:|---:|
| Overall validation | 0.8369 | 0.8210 |
| English | 0.8594 | 0.8394 |
| Hindi | 0.8308 | 0.8140 |
| Kannada | 0.8205 | 0.8043 |

Training completed in 61.8 seconds. The solver converged after 115 iterations. The test set has not been evaluated.

## Repository structure

- `configs/`: experiment configurations
- `docs/`: documentation
- `notebooks/`: Colab notebooks
- `results/metrics/`: evaluation metrics
- `results/figures/`: plots
- `src/`: reusable project code
- `ui/`: final interface

Datasets, secrets, environments, and model checkpoints are excluded from Git.

## Current status

- [x] Dataset verification
- [x] Leakage-safe OOS split
- [x] TF-IDF validation baseline
- [ ] XLM-R multilingual model
- [ ] Noise robustness evaluation
- [ ] Transliteration diagnostic
- [ ] OOS detection and calibration
- [ ] Final test evaluation
- [ ] Professional interface

Only verified experimental results are reported.

Overwriting /content/robust-multilingual-intent-classification/README.md


In [24]:
%cd /content/robust-multilingual-intent-classification
!git status --short
!git diff --check
!git diff --stat

/content/robust-multilingual-intent-classification
 M README.md
?? results/metrics/tfidf_validation_metrics.json
 README.md | 68 +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++--
 1 file changed, 66 insertions(+), 2 deletions(-)


In [25]:
from google.colab import userdata
from pathlib import Path
import os
import subprocess
import tempfile

repo = Path("/content/robust-multilingual-intent-classification")
token = userdata.get("GITHUB_TOKEN")

subprocess.run(
    ["git", "add", "README.md",
     "results/metrics/tfidf_validation_metrics.json"],
    cwd=repo,
    check=True,
)

subprocess.run(
    ["git", "commit", "-m",
     "baseline: add verified TF-IDF validation results"],
    cwd=repo,
    check=True,
)

with tempfile.NamedTemporaryFile(
    mode="w", suffix=".sh", delete=False, encoding="utf-8"
) as askpass:
    askpass.write(
        '#!/bin/sh\n'
        'case "$1" in\n'
        '  *Username*) echo "x-access-token" ;;\n'
        '  *) echo "$GITHUB_TOKEN" ;;\n'
        'esac\n'
    )
    askpass_path = askpass.name

try:
    os.chmod(askpass_path, 0o700)

    env = os.environ.copy()
    env["GITHUB_TOKEN"] = token
    env["GIT_ASKPASS"] = askpass_path
    env["GIT_TERMINAL_PROMPT"] = "0"

    subprocess.run(
        ["git", "push", "origin", "main"],
        cwd=repo,
        env=env,
        check=True,
    )
finally:
    os.remove(askpass_path)

latest = subprocess.run(
    ["git", "log", "-1", "--oneline"],
    cwd=repo,
    text=True,
    capture_output=True,
    check=True,
)

status = subprocess.run(
    ["git", "status", "--short", "--branch"],
    cwd=repo,
    text=True,
    capture_output=True,
    check=True,
)

print("Latest commit:", latest.stdout.strip())
print("Status:", status.stdout.strip())

Latest commit: 78f2dfc baseline: add verified TF-IDF validation results
Status: ## main...origin/main


In [26]:
import subprocess
from pathlib import Path

repo = Path("/content/robust-multilingual-intent-classification")

pull = subprocess.run(
    ["git", "pull", "--ff-only", "origin", "main"],
    cwd=repo,
    text=True,
    capture_output=True,
    check=True,
)

status = subprocess.run(
    ["git", "status", "--short", "--branch"],
    cwd=repo,
    text=True,
    capture_output=True,
    check=True,
)

print(pull.stdout.strip())
print(status.stdout.strip())

Updating 78f2dfc..8b06ce1
Fast-forward
 notebooks/phase1_dataset_verification_tfidf.ipynb | 545 +++++++++++++++++++++-
 1 file changed, 543 insertions(+), 2 deletions(-)
## main...origin/main


In [27]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount("/content/drive")

source = Path(
    "/content/robust-multilingual-intent-classification/"
    "models/tfidf_baseline.joblib"
)

destination_dir = Path(
    "/content/drive/MyDrive/"
    "robust_multilingual_intent/artifacts/models"
)
destination_dir.mkdir(parents=True, exist_ok=True)

destination = destination_dir / source.name
shutil.copy2(source, destination)

print("Saved:", destination)
print(f"Size: {destination.stat().st_size / (1024**2):.2f} MB")

Mounted at /content/drive
Saved: /content/drive/MyDrive/robust_multilingual_intent/artifacts/models/tfidf_baseline.joblib
Size: 27.56 MB


In [28]:
import importlib.metadata
import torch

packages = [
    "transformers",
    "datasets",
    "accelerate",
    "scikit-learn",
]

print("Package versions:")

for package in packages:
    try:
        print(f"{package}: {importlib.metadata.version(package)}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

print("\nGPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    properties = torch.cuda.get_device_properties(0)
    print("GPU:", properties.name)
    print(f"GPU memory: {properties.total_memory / (1024**3):.2f} GB")

Package versions:
transformers: 5.20.2
datasets: 4.8.5
accelerate: 1.15.0
scikit-learn: 1.6.1

GPU available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [29]:
import json
from pathlib import Path

repo = Path("/content/robust-multilingual-intent-classification")

label_names = sorted(known_intents)
label2id = {label: index for index, label in enumerate(label_names)}
id2label = {index: label for label, index in label2id.items()}

train_xlmr = train_known[["id", "locale", "utt", "intent"]].copy()
validation_xlmr = validation_known[
    ["id", "locale", "utt", "intent"]
].copy()

train_xlmr["label"] = train_xlmr["intent"].map(label2id)
validation_xlmr["label"] = validation_xlmr["intent"].map(label2id)

label_mapping = {
    "number_of_labels": len(label_names),
    "label2id": label2id,
    "id2label": {str(key): value for key, value in id2label.items()},
}

mapping_path = repo / "configs" / "label_mapping.json"
mapping_path.write_text(
    json.dumps(label_mapping, indent=2),
    encoding="utf-8",
)

print("Training examples:", len(train_xlmr))
print("Validation examples:", len(validation_xlmr))
print("Number of labels:", len(label_names))
print("Missing training label IDs:", train_xlmr["label"].isna().sum())
print("Missing validation label IDs:", validation_xlmr["label"].isna().sum())
print("Held-out intents in training:",
      set(train_xlmr["intent"]) &
      (set(validation_oos_intents) | set(test_oos_intents)))
print("Minimum training examples per intent:",
      train_xlmr["intent"].value_counts().min())
print("Maximum training examples per intent:",
      train_xlmr["intent"].value_counts().max())
print("Saved:", mapping_path)

Training examples: 26193
Validation examples: 4629
Number of labels: 50
Missing training label IDs: 0
Missing validation label IDs: 0
Held-out intents in training: set()
Minimum training examples per intent: 12
Maximum training examples per intent: 2430
Saved: /content/robust-multilingual-intent-classification/configs/label_mapping.json


In [30]:
intent_counts = (
    train_xlmr.groupby("intent")
    .size()
    .sort_values()
)

print("10 smallest training intents:")
print(intent_counts.head(10).to_string())

print("\n10 largest training intents:")
print(intent_counts.tail(10).to_string())

print("\nCounts for the smallest intent by language:")
smallest_intent = intent_counts.index[0]

print(
    train_xlmr[
        train_xlmr["intent"] == smallest_intent
    ]["locale"].value_counts().sort_index().to_string()
)

print("\nSmallest intent:", smallest_intent)

10 smallest training intents:
intent
cooking_query          12
music_dislikeness      42
audio_volume_other     54
iot_hue_lighton        66
general_greet          75
iot_wemo_on           144
music_settings        153
audio_volume_down     156
datetime_convert      156
iot_wemo_off          156

10 largest training intents:
intent
transport_query     681
qa_definition       801
social_post         849
play_radio          849
datetime_query     1050
news_query         1509
qa_factoid         1632
general_quirky     1665
weather_query      1719
calendar_set       2430

Counts for the smallest intent by language:
locale
en-US    4
hi-IN    4
kn-IN    4

Smallest intent: cooking_query


In [31]:
import numpy as np
from transformers import AutoTokenizer

model_name = "FacebookAI/xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

encoded_lengths = tokenizer(
    train_xlmr["utt"].tolist(),
    add_special_tokens=True,
    truncation=False,
    padding=False,
    return_length=True,
)["length"]

encoded_lengths = np.array(encoded_lengths)

print("Tokenizer:", model_name)
print("Training utterances checked:", len(encoded_lengths))
print("Median token length:", int(np.percentile(encoded_lengths, 50)))
print("95th percentile:", int(np.percentile(encoded_lengths, 95)))
print("99th percentile:", int(np.percentile(encoded_lengths, 99)))
print("Maximum token length:", int(encoded_lengths.max()))
print("Examples longer than 64 tokens:", int((encoded_lengths > 64).sum()))

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

Tokenizer: FacebookAI/xlm-roberta-base
Training utterances checked: 26193
Median token length: 11
95th percentile: 21
99th percentile: 27
Maximum token length: 58
Examples longer than 64 tokens: 0


In [32]:
from datasets import Dataset
from transformers import DataCollatorWithPadding

train_dataset = Dataset.from_pandas(
    train_xlmr.rename(columns={"label": "labels"}),
    preserve_index=False,
)

validation_dataset = Dataset.from_pandas(
    validation_xlmr.rename(columns={"label": "labels"}),
    preserve_index=False,
)

def tokenize_batch(batch):
    return tokenizer(
        batch["utt"],
        truncation=True,
        max_length=64,
    )

columns_to_remove = ["id", "locale", "utt", "intent"]

tokenized_train = train_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=columns_to_remove,
)

tokenized_validation = validation_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=columns_to_remove,
)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True,
    return_tensors="pt",
)

print("Tokenized training rows:", len(tokenized_train))
print("Tokenized validation rows:", len(tokenized_validation))
print("Training columns:", tokenized_train.column_names)
print("First label ID:", tokenized_train[0]["labels"])
print("First token count:", len(tokenized_train[0]["input_ids"]))
print("Label is within 0–49:",
      0 <= tokenized_train[0]["labels"] < 50)

Map:   0%|          | 0/26193 [00:00<?, ? examples/s]

Map:   0%|          | 0/4629 [00:00<?, ? examples/s]

Tokenized training rows: 26193
Tokenized validation rows: 4629
Training columns: ['labels', 'input_ids', 'attention_mask']
First label ID: 1
First token count: 12
Label is within 0–49: True


In [33]:
import torch
from transformers import AutoModelForSequenceClassification, set_seed

set_seed(42)

xlmr_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=50,
    label2id=label2id,
    id2label=id2label,
)

xlmr_model.to("cuda")

sample_batch = data_collator(
    [tokenized_train[index] for index in range(8)]
)
sample_batch = {
    key: value.to("cuda")
    for key, value in sample_batch.items()
}

xlmr_model.eval()

with torch.no_grad():
    sample_output = xlmr_model(**sample_batch)

print("Logits shape:", tuple(sample_output.logits.shape))
print("Expected shape:", (8, 50))
print("Loss is finite:", torch.isfinite(sample_output.loss).item())
print("Sample loss:", float(sample_output.loss))
print(
    "GPU memory allocated:",
    f"{torch.cuda.memory_allocated() / (1024**3):.2f} GB"
)

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Logits shape: (8, 50)
Expected shape: (8, 50)
Loss is finite: True
Sample loss: 4.160451889038086
GPU memory allocated: 1.05 GB


In [35]:
import inspect
from transformers import TrainingArguments

signature = inspect.signature(TrainingArguments.__init__)

print(signature)

print("\nRelevant available arguments:")
for name in signature.parameters:
    if any(
        keyword in name
        for keyword in [
            "warmup",
            "eval",
            "save",
            "batch",
            "epoch",
            "learning",
            "weight",
            "fp16",
        ]
    ):
        print("-", name)

(self, output_dir: str | None = None, per_device_train_batch_size: int = 8, num_train_epochs: float = 3.0, max_steps: int = -1, learning_rate: float = 5e-05, lr_scheduler_type: transformers.trainer_utils.SchedulerType | str = 'linear', lr_scheduler_kwargs: dict | str | None = None, warmup_steps: float = 0, optim: transformers.training_args.OptimizerNames | str = 'adamw_torch_fused', optim_args: str | None = None, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, optim_target_modules: None | str | list[str] = None, gradient_accumulation_steps: int = 1, average_tokens_across_devices: bool = True, max_grad_norm: float = 1.0, label_smoothing_factor: float = 0.0, bf16: bool = False, fp16: bool = False, bf16_full_eval: bool = False, fp16_full_eval: bool = False, tf32: bool | None = None, gradient_checkpointing: bool = False, gradient_checkpointing_kwargs: dict[str, typing.Any] | str | None = None, torch_compile: bool = False, torch_co

In [36]:
import gc
import numpy as np
import torch
from sklearn.metrics import accuracy_score, f1_score
from transformers import Trainer, TrainingArguments

# Remove the temporary forward-pass batch from GPU memory.
for variable_name in ["sample_batch", "sample_output"]:
    if variable_name in globals():
        del globals()[variable_name]

gc.collect()
torch.cuda.empty_cache()

def compute_xlmr_metrics(evaluation_prediction):
    logits, labels = evaluation_prediction
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(
            labels,
            predictions,
            average="macro",
            zero_division=0,
        ),
    }

training_args = TrainingArguments(
    output_dir=(
        "/content/robust-multilingual-intent-classification/"
        "models/xlmr_clean"
    ),
    num_train_epochs=2,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=328,
    optim="adamw_torch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=1,
    fp16=True,
    logging_steps=100,
    report_to="none",
    seed=42,
    data_seed=42,
)

trainer = Trainer(
    model=xlmr_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,
    data_collator=data_collator,
    compute_metrics=compute_xlmr_metrics,
    processing_class=tokenizer,
)

steps_per_epoch = int(
    np.ceil(
        len(tokenized_train) /
        training_args.per_device_train_batch_size
    )
)

print("Trainer created successfully")
print("Epochs:", training_args.num_train_epochs)
print("Batch size:", training_args.per_device_train_batch_size)
print("Steps per epoch:", steps_per_epoch)
print("Approximate total steps:",
      steps_per_epoch * int(training_args.num_train_epochs))
print("Warmup steps:", training_args.warmup_steps)
print("FP16 enabled:", training_args.fp16)
print("GPU memory currently allocated:",
      f"{torch.cuda.memory_allocated() / (1024**3):.2f} GB")
print("Test set used:", False)

Trainer created successfully
Epochs: 2
Batch size: 16
Steps per epoch: 1638
Approximate total steps: 3276
Warmup steps: 328
FP16 enabled: True
GPU memory currently allocated: 1.05 GB
Test set used: False


In [37]:
import time

training_start = time.time()

train_result = trainer.train()

xlmr_training_seconds = time.time() - training_start

print("\nTraining completed.")
print(f"Wall-clock time: {xlmr_training_seconds / 60:.1f} minutes")
print("Training metrics:", train_result.metrics)
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation macro-F1:", trainer.state.best_metric)

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.993797,0.834805,0.791748,0.644493
2,0.617946,0.640998,0.845323,0.762466


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training completed.
Wall-clock time: 12.3 minutes
Training metrics: {'train_runtime': 730.6797, 'train_samples_per_second': 71.695, 'train_steps_per_second': 4.483, 'total_flos': 600434504413320.0, 'train_loss': 1.3001708809709374, 'epoch': 2.0}
Best checkpoint: /content/robust-multilingual-intent-classification/models/xlmr_clean/checkpoint-3276
Best validation macro-F1: 0.7624657472558254


In [38]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

validation_output = trainer.predict(tokenized_validation)
xlmr_validation_predictions = np.argmax(
    validation_output.predictions,
    axis=-1,
)

xlmr_validation_accuracy = accuracy_score(
    validation_xlmr["label"],
    xlmr_validation_predictions,
)

xlmr_validation_macro_f1 = f1_score(
    validation_xlmr["label"],
    xlmr_validation_predictions,
    average="macro",
    zero_division=0,
)

print(f"Overall accuracy: {xlmr_validation_accuracy:.4f}")
print(f"Overall macro-F1: {xlmr_validation_macro_f1:.4f}")

print("\nPer-language validation results:")

xlmr_per_language = {}

for locale in ["en-US", "hi-IN", "kn-IN"]:
    mask = validation_xlmr["locale"].to_numpy() == locale

    locale_accuracy = accuracy_score(
        validation_xlmr.loc[mask, "label"],
        xlmr_validation_predictions[mask],
    )

    locale_macro_f1 = f1_score(
        validation_xlmr.loc[mask, "label"],
        xlmr_validation_predictions[mask],
        average="macro",
        zero_division=0,
    )

    xlmr_per_language[locale] = {
        "accuracy": float(locale_accuracy),
        "macro_f1": float(locale_macro_f1),
    }

    print(
        f"{locale}: accuracy={locale_accuracy:.4f}, "
        f"macro-F1={locale_macro_f1:.4f}"
    )

print("\nTest set evaluated:", False)

Overall accuracy: 0.8453
Overall macro-F1: 0.7625

Per-language validation results:
en-US: accuracy=0.8652, macro-F1=0.7899
hi-IN: accuracy=0.8464, macro-F1=0.7559
kn-IN: accuracy=0.8244, macro-F1=0.7307

Test set evaluated: False


In [39]:
import json
import shutil
from pathlib import Path
import transformers

repo = Path("/content/robust-multilingual-intent-classification")

# Save the best model loaded by Trainer.
local_model_dir = repo / "models" / "xlmr_clean_final"
trainer.save_model(local_model_dir)
tokenizer.save_pretrained(local_model_dir)

xlmr_metrics = {
    "model": "FacebookAI/xlm-roberta-base",
    "training_type": "standard_cross_entropy",
    "random_seed": 42,
    "epochs": 2,
    "learning_rate": 2e-5,
    "train_batch_size": 16,
    "maximum_sequence_length": 64,
    "training_examples": len(train_xlmr),
    "validation_examples": len(validation_xlmr),
    "training_minutes": xlmr_training_seconds / 60,
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "best_epoch": 2,
    "validation_accuracy": float(xlmr_validation_accuracy),
    "validation_macro_f1": float(xlmr_validation_macro_f1),
    "per_language": xlmr_per_language,
    "tfidf_validation_macro_f1": float(validation_macro_f1),
    "macro_f1_difference_vs_tfidf": float(
        xlmr_validation_macro_f1 - validation_macro_f1
    ),
    "transformers_version": transformers.__version__,
    "test_set_evaluated": False,
}

metrics_path = repo / "results" / "metrics" / "xlmr_clean_validation_metrics.json"
metrics_path.write_text(
    json.dumps(xlmr_metrics, indent=2),
    encoding="utf-8",
)

# Store the large model outside GitHub.
drive_model_dir = Path(
    "/content/drive/MyDrive/"
    "robust_multilingual_intent/artifacts/models/xlmr_clean_final"
)

if drive_model_dir.exists():
    shutil.rmtree(drive_model_dir)

shutil.copytree(local_model_dir, drive_model_dir)

total_size = sum(
    path.stat().st_size
    for path in drive_model_dir.rglob("*")
    if path.is_file()
)

print("Metrics saved:", metrics_path)
print("Model saved to Drive:", drive_model_dir)
print(f"Drive model size: {total_size / (1024**3):.2f} GB")
print("Test set evaluated:", False)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Metrics saved: /content/robust-multilingual-intent-classification/results/metrics/xlmr_clean_validation_metrics.json
Model saved to Drive: /content/drive/MyDrive/robust_multilingual_intent/artifacts/models/xlmr_clean_final
Drive model size: 1.05 GB
Test set evaluated: False


In [40]:
%cd /content/robust-multilingual-intent-classification
!git status --short --untracked-files=all
!git check-ignore -v models/xlmr_clean_final/model.safetensors
!git diff --check

/content/robust-multilingual-intent-classification
?? configs/label_mapping.json
?? results/metrics/xlmr_clean_validation_metrics.json
.gitignore:224:models/	models/xlmr_clean_final/model.safetensors


In [41]:
from google.colab import userdata
from pathlib import Path
import os
import subprocess
import tempfile

repo = Path("/content/robust-multilingual-intent-classification")
token = userdata.get("GITHUB_TOKEN")

subprocess.run(
    ["git", "add",
     "configs/label_mapping.json",
     "results/metrics/xlmr_clean_validation_metrics.json"],
    cwd=repo,
    check=True,
)

subprocess.run(
    ["git", "commit", "-m",
     "model: add verified clean XLM-R validation results"],
    cwd=repo,
    check=True,
)

with tempfile.NamedTemporaryFile(
    mode="w", suffix=".sh", delete=False, encoding="utf-8"
) as askpass:
    askpass.write(
        '#!/bin/sh\n'
        'case "$1" in\n'
        '  *Username*) echo "x-access-token" ;;\n'
        '  *) echo "$GITHUB_TOKEN" ;;\n'
        'esac\n'
    )
    askpass_path = askpass.name

try:
    os.chmod(askpass_path, 0o700)

    env = os.environ.copy()
    env["GITHUB_TOKEN"] = token
    env["GIT_ASKPASS"] = askpass_path
    env["GIT_TERMINAL_PROMPT"] = "0"

    subprocess.run(
        ["git", "push", "origin", "main"],
        cwd=repo,
        env=env,
        check=True,
    )
finally:
    os.remove(askpass_path)

latest = subprocess.run(
    ["git", "log", "-1", "--oneline"],
    cwd=repo,
    text=True,
    capture_output=True,
    check=True,
)

status = subprocess.run(
    ["git", "status", "--short", "--branch"],
    cwd=repo,
    text=True,
    capture_output=True,
    check=True,
)

print("Latest commit:", latest.stdout.strip())
print("Status:", status.stdout.strip())

Latest commit: f9514f6 model: add verified clean XLM-R validation results
Status: ## main...origin/main


In [42]:
import subprocess
from pathlib import Path

repo = Path("/content/robust-multilingual-intent-classification")

pull = subprocess.run(
    ["git", "pull", "--ff-only", "origin", "main"],
    cwd=repo,
    text=True,
    capture_output=True,
    check=True,
)

status = subprocess.run(
    ["git", "status", "--short", "--branch"],
    cwd=repo,
    text=True,
    capture_output=True,
    check=True,
)

print(pull.stdout.strip())
print(status.stdout.strip())

Updating f9514f6..e2ab604
Fast-forward
 notebooks/phase1_dataset_verification_tfidf.ipynb | 5557 ++++++++++++++++++++-
 1 file changed, 5536 insertions(+), 21 deletions(-)
## main...origin/main


In [43]:
import numpy as np
import pandas as pd
import torch

label_counts = (
    train_xlmr["label"]
    .value_counts()
    .sort_index()
    .reindex(range(50))
)

number_of_examples = len(train_xlmr)
number_of_classes = 50

raw_weights = np.sqrt(
    number_of_examples /
    (number_of_classes * label_counts.to_numpy())
)

# Normalise so the average class weight is 1.
normalised_weights = raw_weights / raw_weights.mean()

class_weights = torch.tensor(
    normalised_weights,
    dtype=torch.float32,
)

weight_report = pd.DataFrame({
    "intent": [id2label[index] for index in range(50)],
    "training_examples": label_counts.to_numpy(),
    "class_weight": normalised_weights,
}).sort_values("class_weight", ascending=False)

print("Number of class weights:", len(class_weights))
print(f"Minimum weight: {class_weights.min().item():.4f}")
print(f"Maximum weight: {class_weights.max().item():.4f}")
print(f"Mean weight: {class_weights.mean().item():.4f}")

print("\nHighest-weight rare intents:")
print(weight_report.head(10).to_string(index=False))

print("\nLowest-weight frequent intents:")
print(weight_report.tail(5).to_string(index=False))

Number of class weights: 50
Minimum weight: 0.3305
Maximum weight: 4.7027
Mean weight: 1.0000

Highest-weight rare intents:
            intent  training_examples  class_weight
     cooking_query                 12      4.702686
 music_dislikeness                 42      2.513691
audio_volume_other                 54      2.216867
   iot_hue_lighton                 66      2.005232
     general_greet                 75      1.881074
       iot_wemo_on                144      1.357548
    music_settings                153      1.317015
  datetime_convert                156      1.304290
 audio_volume_down                156      1.304290
      iot_wemo_off                156      1.304290

Lowest-weight frequent intents:
        intent  training_examples  class_weight
    news_query               1509      0.419365
    qa_factoid               1632      0.403252
general_quirky               1665      0.399236
 weather_query               1719      0.392915
  calendar_set               24

In [44]:
import gc
import torch
import torch.nn.functional as F
from transformers import (
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    set_seed,
)

# Release the previous clean model from GPU memory.
for variable_name in [
    "trainer",
    "xlmr_model",
    "validation_output",
]:
    if variable_name in globals():
        del globals()[variable_name]

gc.collect()
torch.cuda.empty_cache()

class ClassWeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,
    ):
        labels = inputs["labels"]

        model_inputs = {
            key: value
            for key, value in inputs.items()
            if key != "labels"
        }

        outputs = model(**model_inputs)
        logits = outputs.logits

        loss = F.cross_entropy(
            logits,
            labels,
            weight=self.class_weights.to(logits.device),
        )

        return (loss, outputs) if return_outputs else loss

set_seed(42)

weighted_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=50,
    label2id=label2id,
    id2label=id2label,
)

weighted_args = TrainingArguments(
    output_dir=(
        "/content/robust-multilingual-intent-classification/"
        "models/xlmr_weighted"
    ),
    num_train_epochs=2,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=328,
    optim="adamw_torch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=1,
    fp16=True,
    logging_steps=100,
    report_to="none",
    seed=42,
    data_seed=42,
)

weighted_trainer = ClassWeightedTrainer(
    model=weighted_model,
    args=weighted_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,
    data_collator=data_collator,
    compute_metrics=compute_xlmr_metrics,
    processing_class=tokenizer,
    class_weights=class_weights,
)

# Verify the custom loss before full training.
check_batch = data_collator(
    [tokenized_train[index] for index in range(8)]
)

check_batch = {
    key: value.to(weighted_model.device)
    for key, value in check_batch.items()
}

weighted_model.eval()

with torch.no_grad():
    check_loss = weighted_trainer.compute_loss(
        weighted_model,
        check_batch,
    )

print("Weighted trainer created successfully")
print("Weighted loss is finite:", torch.isfinite(check_loss).item())
print("Sample weighted loss:", float(check_loss))
print("Epochs:", weighted_args.num_train_epochs)
print("Test set used:", False)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Weighted trainer created successfully
Weighted loss is finite: True
Sample weighted loss: 4.161773681640625
Epochs: 2
Test set used: False


In [45]:
import gc
import time
import torch

if "check_batch" in globals():
    del check_batch
gc.collect()
torch.cuda.empty_cache()

weighted_training_start = time.time()

weighted_train_result = weighted_trainer.train()

weighted_training_seconds = (
    time.time() - weighted_training_start
)

print("\nWeighted training completed.")
print(
    "Wall-clock time:",
    f"{weighted_training_seconds / 60:.1f} minutes",
)
print("Training metrics:", weighted_train_result.metrics)
print(
    "Best checkpoint:",
    weighted_trainer.state.best_model_checkpoint,
)
print(
    "Best validation macro-F1:",
    weighted_trainer.state.best_metric,
)

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,1.032274,0.858322,0.812054,0.733888
2,0.658845,0.657302,0.852236,0.766656


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Weighted training completed.
Wall-clock time: 12.4 minutes
Training metrics: {'train_runtime': 741.6387, 'train_samples_per_second': 70.635, 'train_steps_per_second': 4.417, 'total_flos': 600434504413320.0, 'train_loss': 1.3811232935960185, 'epoch': 2.0}
Best checkpoint: /content/robust-multilingual-intent-classification/models/xlmr_weighted/checkpoint-3276
Best validation macro-F1: 0.7666557770079698


In [46]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

weighted_validation_output = weighted_trainer.predict(
    tokenized_validation
)

weighted_validation_predictions = np.argmax(
    weighted_validation_output.predictions,
    axis=-1,
)

weighted_validation_accuracy = accuracy_score(
    validation_xlmr["label"],
    weighted_validation_predictions,
)

weighted_validation_macro_f1 = f1_score(
    validation_xlmr["label"],
    weighted_validation_predictions,
    average="macro",
    zero_division=0,
)

weighted_per_language = {}

print(
    f"Overall accuracy: "
    f"{weighted_validation_accuracy:.4f}"
)
print(
    f"Overall macro-F1: "
    f"{weighted_validation_macro_f1:.4f}"
)

print("\nPer-language validation results:")

for locale in ["en-US", "hi-IN", "kn-IN"]:
    mask = validation_xlmr["locale"].to_numpy() == locale

    locale_accuracy = accuracy_score(
        validation_xlmr.loc[mask, "label"],
        weighted_validation_predictions[mask],
    )

    locale_macro_f1 = f1_score(
        validation_xlmr.loc[mask, "label"],
        weighted_validation_predictions[mask],
        average="macro",
        zero_division=0,
    )

    weighted_per_language[locale] = {
        "accuracy": float(locale_accuracy),
        "macro_f1": float(locale_macro_f1),
    }

    print(
        f"{locale}: accuracy={locale_accuracy:.4f}, "
        f"macro-F1={locale_macro_f1:.4f}"
    )

print("\nMacro-F1 comparisons:")
print(f"TF-IDF: {validation_macro_f1:.4f}")
print(f"Clean XLM-R: {xlmr_validation_macro_f1:.4f}")
print(f"Weighted XLM-R: {weighted_validation_macro_f1:.4f}")
print("Test set evaluated:", False)

Overall accuracy: 0.8522
Overall macro-F1: 0.7667

Per-language validation results:
en-US: accuracy=0.8717, macro-F1=0.8027
hi-IN: accuracy=0.8496, macro-F1=0.7770
kn-IN: accuracy=0.8354, macro-F1=0.7443

Macro-F1 comparisons:
TF-IDF: 0.8210
Clean XLM-R: 0.7625
Weighted XLM-R: 0.7667
Test set evaluated: False


In [47]:
import json
import shutil
from pathlib import Path
import transformers

repo = Path("/content/robust-multilingual-intent-classification")

local_weighted_dir = repo / "models" / "xlmr_weighted_final"
weighted_trainer.save_model(local_weighted_dir)
tokenizer.save_pretrained(local_weighted_dir)

weighting_config = {
    "method": "square_root_inverse_frequency",
    "normalisation": "mean weight equals 1",
    "formula": "sqrt(N / (K * class_count)), then divide by mean",
    "number_of_classes": 50,
    "weights": {
        id2label[index]: {
            "training_examples": int(label_counts.iloc[index]),
            "weight": float(class_weights[index]),
        }
        for index in range(50)
    },
}

weighting_path = repo / "configs" / "xlmr_class_weights.json"
weighting_path.write_text(
    json.dumps(weighting_config, indent=2),
    encoding="utf-8",
)

weighted_metrics = {
    "model": "FacebookAI/xlm-roberta-base",
    "training_type": "square_root_inverse_frequency_weighted_cross_entropy",
    "random_seed": 42,
    "epochs": 2,
    "learning_rate": 2e-5,
    "train_batch_size": 16,
    "maximum_sequence_length": 64,
    "training_examples": len(train_xlmr),
    "validation_examples": len(validation_xlmr),
    "training_minutes": weighted_training_seconds / 60,
    "best_checkpoint": weighted_trainer.state.best_model_checkpoint,
    "best_epoch": 2,
    "validation_accuracy": float(weighted_validation_accuracy),
    "validation_macro_f1": float(weighted_validation_macro_f1),
    "per_language": weighted_per_language,
    "clean_xlmr_macro_f1": float(xlmr_validation_macro_f1),
    "tfidf_macro_f1": float(validation_macro_f1),
    "improvement_vs_clean_xlmr": float(
        weighted_validation_macro_f1 -
        xlmr_validation_macro_f1
    ),
    "difference_vs_tfidf": float(
        weighted_validation_macro_f1 -
        validation_macro_f1
    ),
    "transformers_version": transformers.__version__,
    "test_set_evaluated": False,
}

weighted_metrics_path = (
    repo / "results" / "metrics" /
    "xlmr_weighted_validation_metrics.json"
)

weighted_metrics_path.write_text(
    json.dumps(weighted_metrics, indent=2),
    encoding="utf-8",
)

drive_weighted_dir = Path(
    "/content/drive/MyDrive/"
    "robust_multilingual_intent/artifacts/models/"
    "xlmr_weighted_final"
)

if drive_weighted_dir.exists():
    shutil.rmtree(drive_weighted_dir)

shutil.copytree(local_weighted_dir, drive_weighted_dir)

total_size = sum(
    path.stat().st_size
    for path in drive_weighted_dir.rglob("*")
    if path.is_file()
)

print("Weighting configuration:", weighting_path)
print("Metrics:", weighted_metrics_path)
print("Model saved to Drive:", drive_weighted_dir)
print(f"Drive model size: {total_size / (1024**3):.2f} GB")
print("Test set evaluated:", False)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Weighting configuration: /content/robust-multilingual-intent-classification/configs/xlmr_class_weights.json
Metrics: /content/robust-multilingual-intent-classification/results/metrics/xlmr_weighted_validation_metrics.json
Model saved to Drive: /content/drive/MyDrive/robust_multilingual_intent/artifacts/models/xlmr_weighted_final
Drive model size: 1.05 GB
Test set evaluated: False


In [48]:
%cd /content/robust-multilingual-intent-classification
!git status --short --untracked-files=all
!git check-ignore -v models/xlmr_weighted_final/model.safetensors
!git diff --check

/content/robust-multilingual-intent-classification
?? configs/xlmr_class_weights.json
?? results/metrics/xlmr_weighted_validation_metrics.json
.gitignore:224:models/	models/xlmr_weighted_final/model.safetensors


In [49]:
from google.colab import userdata
from pathlib import Path
import os
import subprocess
import tempfile

repo = Path("/content/robust-multilingual-intent-classification")
token = userdata.get("GITHUB_TOKEN")

subprocess.run(
    ["git", "add",
     "configs/xlmr_class_weights.json",
     "results/metrics/xlmr_weighted_validation_metrics.json"],
    cwd=repo,
    check=True,
)

subprocess.run(
    ["git", "commit", "-m",
     "model: evaluate class-weighted XLM-R baseline"],
    cwd=repo,
    check=True,
)

with tempfile.NamedTemporaryFile(
    mode="w", suffix=".sh", delete=False, encoding="utf-8"
) as askpass:
    askpass.write(
        '#!/bin/sh\n'
        'case "$1" in\n'
        '  *Username*) echo "x-access-token" ;;\n'
        '  *) echo "$GITHUB_TOKEN" ;;\n'
        'esac\n'
    )
    askpass_path = askpass.name

try:
    os.chmod(askpass_path, 0o700)

    env = os.environ.copy()
    env["GITHUB_TOKEN"] = token
    env["GIT_ASKPASS"] = askpass_path
    env["GIT_TERMINAL_PROMPT"] = "0"

    subprocess.run(
        ["git", "push", "origin", "main"],
        cwd=repo,
        env=env,
        check=True,
    )
finally:
    os.remove(askpass_path)

latest = subprocess.run(
    ["git", "log", "-1", "--oneline"],
    cwd=repo,
    text=True,
    capture_output=True,
    check=True,
)

status = subprocess.run(
    ["git", "status", "--short", "--branch"],
    cwd=repo,
    text=True,
    capture_output=True,
    check=True,
)

print("Latest commit:", latest.stdout.strip())
print("Status:", status.stdout.strip())

Latest commit: e24c79d model: evaluate class-weighted XLM-R baseline
Status: ## main...origin/main


In [50]:
from pathlib import Path

repo = Path("/content/robust-multilingual-intent-classification")
readme_path = repo / "README.md"
readme = readme_path.read_text(encoding="utf-8")

phase2_section = """
## Phase 2 verified multilingual models

Two XLM-R-base models were fine-tuned for two epochs using the same 50 known intents.

| Model | Validation accuracy | Validation macro-F1 |
|---|---:|---:|
| TF-IDF baseline | 0.8369 | **0.8210** |
| Clean XLM-R | 0.8453 | 0.7625 |
| Weighted XLM-R | **0.8522** | 0.7667 |

Weighted XLM-R improved over clean XLM-R across all three languages:

| Language | Clean XLM-R macro-F1 | Weighted XLM-R macro-F1 |
|---|---:|---:|
| English | 0.7899 | **0.8027** |
| Hindi | 0.7559 | **0.7770** |
| Kannada | 0.7307 | **0.7443** |

TF-IDF remains the strongest clean-text model by macro-F1. Weighted XLM-R is retained for later robustness and transliteration comparisons. Large checkpoints are stored outside GitHub in Google Drive.

"""

if "## Phase 2 verified multilingual models" not in readme:
    readme = readme.replace(
        "## Repository structure",
        phase2_section + "## Repository structure",
    )

readme = readme.replace(
    "- [ ] XLM-R multilingual model",
    "- [x] XLM-R multilingual model",
)

readme_path.write_text(readme, encoding="utf-8")

print("README updated.")
print("\nGit status:")

README updated.

Git status:


In [52]:
%cd /content/robust-multilingual-intent-classification
!git status --short
!git diff --check
!git diff --stat

/content/robust-multilingual-intent-classification
 M README.md
 README.md | 23 ++++++++++++++++++++++-
 1 file changed, 22 insertions(+), 1 deletion(-)


In [53]:
from google.colab import userdata
from pathlib import Path
import os
import subprocess
import tempfile

repo = Path("/content/robust-multilingual-intent-classification")
token = userdata.get("GITHUB_TOKEN")

subprocess.run(
    ["git", "add", "README.md"],
    cwd=repo,
    check=True,
)

subprocess.run(
    ["git", "commit", "-m",
     "docs: report verified XLM-R validation results"],
    cwd=repo,
    check=True,
)

with tempfile.NamedTemporaryFile(
    mode="w", suffix=".sh", delete=False, encoding="utf-8"
) as askpass:
    askpass.write(
        '#!/bin/sh\n'
        'case "$1" in\n'
        '  *Username*) echo "x-access-token" ;;\n'
        '  *) echo "$GITHUB_TOKEN" ;;\n'
        'esac\n'
    )
    askpass_path = askpass.name

try:
    os.chmod(askpass_path, 0o700)

    env = os.environ.copy()
    env["GITHUB_TOKEN"] = token
    env["GIT_ASKPASS"] = askpass_path
    env["GIT_TERMINAL_PROMPT"] = "0"

    subprocess.run(
        ["git", "push", "origin", "main"],
        cwd=repo,
        env=env,
        check=True,
    )
finally:
    os.remove(askpass_path)

latest = subprocess.run(
    ["git", "log", "-1", "--oneline"],
    cwd=repo,
    text=True,
    capture_output=True,
    check=True,
)

status = subprocess.run(
    ["git", "status", "--short", "--branch"],
    cwd=repo,
    text=True,
    capture_output=True,
    check=True,
)

print("Latest commit:", latest.stdout.strip())
print("Status:", status.stdout.strip())

Latest commit: f7f58af docs: report verified XLM-R validation results
Status: ## main...origin/main
